# Mixed reduced GP

continuous feature だけをPCA/PLSで圧縮し、categorical feature を native categorical covariance に残します。


## このNotebookで学ぶこと

- Mixed入力の**連続変数**と**カテゴリ変数**を区別する
- `MixedPCAGP` / `MixedPLSGP` で連続部分だけを次元削減する
- `model.fit()` によるGP学習とPosteriorの可視化
- 事後平均による有限候補選択と、**qLogEI + `optimize_acqf_mixed()`** による獲得関数最適化の違い
- 提案した候補を元のMixed空間の目的関数で評価する

**重要：** PCA/PLSの潜在次元は2ですが、候補探索は元の5つの連続変数とカテゴリ値を対象にします。カテゴリ値0・1・2を連続値として補間しません。

## Mixedデータ


### 入力データの構造

各行は `[x0, x1, x2, x3, x4, category]` の6列です。先頭5列は `[0, 1]` の連続変数、末尾は `{0, 1, 2}` のカテゴリ変数です。

この例の合成目的関数は `sin(4*x0) + 0.2*category` です。カテゴリは**数値の大小関係を仮定した連続変数ではなく**、離散ラベルとして扱います。

In [ ]:
import torch
from robotorchan.models import MixedPCAGP, MixedPLSGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
continuous = torch.rand(18, 5)
category = torch.randint(0, 3, (18, 1)).double()
X = torch.cat([continuous, category], dim=-1)
Y = torch.sin(4 * continuous[:, :1]) + 0.2 * category


## Mixed reduction


### PCAとPLSの違い

PCAは連続入力の分散が大きい方向を利用します。PLSは目的変数との関係を考慮した方向を利用します。どちらもカテゴリ列は削減せず、Mixed GPのカテゴリ共分散に渡します。

**注意：** PCA/PLSでモデル内部の入力次元を下げても、元の連続5次元空間の候補最適化が自動的に2次元になるわけではありません。

In [ ]:
pca = MixedPCAGP(X, Y, n_components=2, cat_dims=[5])
pls = MixedPLSGP(X, Y, n_components=2, cat_dims=[5])
print(pca.raw_train_X.shape, pls.raw_train_X.shape)


## Original mixed-space posterior


In [ ]:
X_test = torch.cat([torch.rand(3, 5), torch.tensor([[0.0], [1.0], [2.0]])], dim=-1)
print(pca.posterior(X_test).mean.squeeze(-1))
print(pls.posterior(X_test).mean.squeeze(-1))


### 学習前後のPosterior

上の `posterior()` は学習前の呼び出し例です。次の `fit()` でGPのハイパーパラメータを学習した後に、予測平均と不確実性を可視化します。

## GP fitting

PCA/PLSは連続変数のみを削減し、カテゴリ列は保持します。既存のMLLで両モデルを学習します。


In [ ]:

models = {"MixedPCA": pca, "MixedPLS": pls}
for name, model in models.items():
    model.fit()
    model.eval()
    print(name, "fitted")


### カテゴリ別の予測平均と不確実性

先頭の連続変数 `x0` だけを変え、残り4列を0.5に固定した断面を表示します。カテゴリ0・1・2の予測を並べ、PCAとPLSの違いを確認します。

帯は潜在関数の事後標準偏差の約1.96倍です。観測ノイズを含む区間ではなく、6次元Mixed空間全体の精度を示すものでもありません。

In [ ]:
import matplotlib.pyplot as plt

slice_features = torch.full((101, 5), 0.5)
slice_features[:, 0] = torch.linspace(0.0, 1.0, len(slice_features))
fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True, sharey=True)
for row, (name, surrogate) in enumerate(models.items()):
    surrogate.eval()
    for cat in (0, 1, 2):
        ax = axes[row, cat]
        test_mixed = torch.cat(
            [slice_features, torch.full((len(slice_features), 1), float(cat))],
            dim=-1,
        )
        with torch.no_grad():
            post = surrogate.posterior(test_mixed)
            mean = post.mean.squeeze(-1)
            std = post.variance.clamp_min(0).sqrt().squeeze(-1)
        true_y = torch.sin(4 * slice_features[:, 0]) + 0.2 * cat
        ax.plot(slice_features[:, 0], true_y, "--", label="true slice")
        ax.plot(slice_features[:, 0], mean, label="posterior mean")
        ax.fill_between(slice_features[:, 0], mean - 1.96 * std,
                        mean + 1.96 * std, alpha=0.2,
                        label="approx. 95% latent interval")
        ax.set_title(f"{name} / category {cat}")
        ax.set_xlabel("x0 (x1..x4 = 0.5)")
    axes[row, 0].set_ylabel("objective")
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.show()

### PCAとPLSのカテゴリ別予測残差

上の6枚のグラフに加えて、同じ断面の**RMSE（予測平均と合成関数の真値との差）**をカテゴリ0・1・2ごとに集計します。これは可視化した断面上の誤差であり、6次元全体の汎化誤差や手法の優劣ではありません。

縦軸が小さいほど、この断面では真値に近い予測です。カテゴリのIDを連続量として補間せず、それぞれ別に評価します。


In [ ]:
categories = (0, 1, 2)
rmse_by_model = {}
for name, surrogate in models.items():
    errors = []
    for category_id in categories:
        query = torch.cat(
            [
                slice_features,
                torch.full((len(slice_features), 1), float(category_id)),
            ],
            dim=-1,
        )
        with torch.no_grad():
            predicted = surrogate.posterior(query).mean.squeeze(-1)
        truth = torch.sin(4 * slice_features[:, 0]) + 0.2 * category_id
        errors.append((predicted - truth).square().mean().sqrt().item())
    rmse_by_model[name] = errors

fig, ax = plt.subplots(figsize=(8, 4))
positions = torch.arange(len(categories), dtype=torch.double).numpy()
width = 0.35
for index, (name, errors) in enumerate(rmse_by_model.items()):
    ax.bar(positions + (index - 0.5) * width, errors, width, label=name)
ax.set_xticks(positions, [f"Category {cat}" for cat in categories])
ax.set_ylabel("Slice RMSE")
ax.set_title("Original-space slice error by categorical level")
ax.legend()
plt.tight_layout()
plt.show()

## Mixed-space candidate proposal and evaluation

カテゴリ値0・1・2を明示的に列挙し、元の5連続変数と組み合わせた候補プールからposterior mean最大の点を選びます。


In [ ]:
def objective(candidate):
    return torch.sin(4 * candidate[..., :1]) + 0.2 * candidate[..., 5:6]


continuous_pool = torch.rand(64, 5)
candidate_pool = torch.cat(
    [
        torch.cat(
            [continuous_pool, torch.full((64, 1), float(category))],
            dim=-1,
        )
        for category in (0, 1, 2)
    ],
    dim=0,
)
for name, model in models.items():
    with torch.no_grad():
        scores = model.posterior(candidate_pool).mean.squeeze(-1)
    candidate = candidate_pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == torch.Size([1, 6])
    assert candidate[0, 5].item() in (0, 1, 2)
    print(name, "candidate:", candidate)
    print(name, "posterior mean:", scores.max())
    print(name, "objective:", objective(candidate))


### 事後平均最大化と獲得関数最適化

次の既存例は192個の有効なMixed候補の**事後平均**を比較します。これは活用（exploitation）中心の有限プール探索です。

続く例では、BoTorch標準の `qLogExpectedImprovement` と `optimize_acqf_mixed()` を使用します。カテゴリ0・1・2を `fixed_features_list` に列挙し、それぞれのカテゴリで連続変数を最適化します。候補は元の6次元座標で返されます。

### qLogEIによるMixed候補最適化

`qLogEI` は予測分布に基づく改善の期待値を評価します。`optimize_acqf_mixed` はカテゴリごとに連続部分を最適化し、最も獲得関数値が高い候補を返します。

カテゴリ値を最適化中に連続的に動かすことはせず、カテゴリ集合 `{0,1,2}` を明示します。`best_f` は学習データの最大観測値です。

In [ ]:
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf_mixed
from botorch.sampling.normal import SobolQMCNormalSampler

mixed_qlogei_proposals = {}
mixed_bounds = torch.stack([torch.zeros(6), torch.tensor([1., 1., 1., 1., 1., 2.])])
for name, surrogate in models.items():
    acqf = qLogExpectedImprovement(
        model=surrogate,
        best_f=Y.max(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32])),
    )
    proposed, acq_value = optimize_acqf_mixed(
        acq_function=acqf,
        bounds=mixed_bounds,
        q=1,
        num_restarts=3,
        raw_samples=32,
        fixed_features_list=[{5: float(cat)} for cat in (0, 1, 2)],
    )
    mixed_qlogei_proposals[name] = proposed.detach().clone()
    assert proposed.shape == (1, 6)
    assert proposed[0, 5].item() in (0, 1, 2)
    assert torch.all(proposed[:, :5] >= 0)
    assert torch.all(proposed[:, :5] <= 1)
    print(name, "qLogEI candidate:", proposed)
    print(name, "acquisition:", acq_value)
    print(name, "objective:", objective(proposed))

### qLogEIで提案されたカテゴリと目的値

`optimize_acqf_mixed` で提案した候補の**カテゴリ値**と、合成目的関数で評価した**実際の応答**を並べます。各モデルの目的関数値は同じ式で評価しますが、選ばれた候補の他の連続変数は異なります。単一回の提案結果から探索性能全体の優劣は判断できません。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
proposal_names = list(mixed_qlogei_proposals)
selected_categories = [
    mixed_qlogei_proposals[name][0, 5].item() for name in proposal_names
]
evaluated_values = [
    objective(mixed_qlogei_proposals[name]).item() for name in proposal_names
]
axes[0].bar(proposal_names, selected_categories)
axes[0].set_ylabel("Selected category")
axes[0].set_yticks([0, 1, 2])
axes[1].bar(proposal_names, evaluated_values)
axes[1].set_ylabel("Evaluated objective")
for ax in axes:
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## まとめ

| 処理 | 方法 |
|---|---|
| Mixed入力 | 5つの連続変数 + 1つのカテゴリ変数 |
| モデル | `MixedPCAGP` / `MixedPLSGP` |
| GP学習 | `model.fit()` |
| 予測 | 元のMixed座標で `posterior()` |
| 比較例 | 有限候補プールの事後平均最大化 |
| BO探索 | `qLogExpectedImprovement` + `optimize_acqf_mixed()` |
| 候補評価 | 元のMixed空間で `objective()` |

このNotebookは学習済みモデルからの**1回の候補提案と評価**までを扱います。反復的なBOループやベンチマークではありません。